In [1]:
from transformers import BartForConditionalGeneration, BartTokenizer, AutoTokenizer
from datasets import Dataset
import json
import torch



c:\All\Maxym\Projects\QA-SLM\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("../eval_selection/test_dataset_40.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
input_pairs = [{"text": item["text"]} for item in data]
print(input_pairs[0]["text"])

# Створення Dataset
dataset = Dataset.from_list(input_pairs)

print("example:", dataset[0])

What does this manual contain?
example: {'text': 'What does this manual contain?'}


In [4]:
model_path = "../models/bart_finetuned_model_last/checkpoint-6400"

FT = True
if FT:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = BartForConditionalGeneration.from_pretrained(model_path)
else:
    tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
    model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model.to(device)

Using device: cuda


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [5]:
SEED = 12345
import random as _random
_random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [6]:
def generate_response(instruction):
    input_text = instruction if instruction.lower().startswith("question:") else f"question: {instruction}"
    inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=512
    )

    # Перекидаємо input_ids та attention_mask на GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output_ids = model.generate(**inputs, max_length=64)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


In [7]:
# Вивід
results = []
for example in dataset:
    instr = example["text"]
    gen = generate_response(instr)
    results.append({"instruction": instr, "generated": gen})

    print("Instruction:", instr)
    print("Generated:", gen)
    print("-" * 50)

Instruction: What does this manual contain?
Generated: It contains important information on how to operate your appliances safely and efficiently.
--------------------------------------------------
Instruction: Why should I read the manual?
Generated: Reading it ensures you know how to safely and efficiently operate your appliances.
--------------------------------------------------
Instruction: What does the “Warning” icon mean?
Generated: It indicates hazards or unsafe practices that may cause severe injury, death, or property damage.
--------------------------------------------------
Instruction: What water pressure should the refrigerator have?
Generated: The water pressure should be between 206-861 kPa (30-125 psi).
--------------------------------------------------
Instruction: How can I replace the LED lamps?
Generated: Contact a local Samsung service centre or its agents to replace the lamps.
--------------------------------------------------
Instruction: What should I do to av

In [16]:
# Збереження результатів у файл
with open("BART_fridge_gptoss-20b_responses.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

In [8]:
# Збереження результатів у txt файл
with open("BART_fridge_gptoss-20b+OpenChat_1.98.txt", "w", encoding="utf-8") as f:
    for item in results:
        f.write(f"Інструкція: {item['instruction']}\n")
        f.write(f"Відповідь: {item['generated']}\n")
        f.write("-" * 50 + "\n")